In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
from ipywidgets import interactive, FloatSlider, RadioButtons, HBox, VBox, Layout, HTML, HTMLMath
from IPython.display import display

# ------------------------------------------------------------
# INTRODUCTION
# ------------------------------------------------------------

intro_output = HTML(value="""
<div style="border:1px solid #b8b8b8; padding:12px 16px; margin:5px 0 15px 0; width:1080px; font-size:14px; line-height:1.55;">

<b>Purpose of this interactive notebook</b><br><br>

This notebook demonstrates the frequency response of first-order active
<b>low-pass</b> and <b>high-pass</b> filters implemented with an operational amplifier.<br><br>

<b>What to observe:</b><br>

&bull; Select a <b>Low-pass</b> or <b>High-pass</b> filter.<br>

&bull; Select a <b>Non-inverting</b> or <b>Inverting</b> operational-amplifier configuration.<br>

&bull; For the non-inverting configuration,
<b>K = 1 + R₂/R₃</b>, whereas for the inverting configuration,
<b>K = −R₂/R₁</b>.<br>

&bull; Changing the RC time constant changes the cutoff frequency and therefore
moves the frequency response <b>horizontally</b> along the fixed logarithmic frequency axis.<br>

&bull; Changing the amplifier gain changes the magnitude level of the response.<br>

&bull; The magnitude and phase curves are updated <b>continuously in real time</b>
while the sliders are moved.<br>

&bull; The vertical dotted line indicates the cutoff frequency ω<sub>c</sub>.
At this frequency the magnitude is 3.01 dB below the corresponding passband gain.

</div>
""")

# ------------------------------------------------------------
# CONTROLS
# ------------------------------------------------------------

filter_title = HTML(value="<b>Filter Type:</b>")
filter_radio = RadioButtons(options=['Low-pass', 'High-pass'], value='Low-pass', description='', layout=Layout(width='150px'))

configuration_title = HTML(value="<b>Configuration:</b>")
configuration_radio = RadioButtons(options=['Non-inverting', 'Inverting'], value='Non-inverting', description='', layout=Layout(width='170px'))

R1 = FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0, description='R₁ (kΩ):', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='320px'))
C1 = FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0, description='C₁ (μF):', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='320px'))
R2 = FloatSlider(min=0.1, max=10.0, step=0.1, value=2.0, description='R₂ (kΩ):', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='320px'))
R3 = FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0, description='R₃ (kΩ):', readout=True, readout_format='.2f', continuous_update=True, layout=Layout(width='320px'))

# ------------------------------------------------------------
# CONTROL LAYOUT
# ------------------------------------------------------------

filter_box = VBox([filter_title, filter_radio], layout=Layout(width='180px'))
configuration_box = VBox([configuration_title, configuration_radio], layout=Layout(width='190px'))

selection_row = HBox([filter_box, configuration_box], layout=Layout(width='420px', justify_content='space-between', align_items='flex-start'))

control_box_1 = VBox([R1, C1], layout=Layout(width='340px'))
control_box_2 = VBox([R2, R3], layout=Layout(width='340px'))

control_row = HBox([control_box_1, control_box_2], layout=Layout(width='720px', justify_content='space-between', align_items='flex-start'))

# ------------------------------------------------------------
# NUMERICAL OUTPUT
# ------------------------------------------------------------

parameter_title = HTML(value="<b>Calculated Filter Parameters</b>")

transfer_output = HTMLMath(layout=Layout(width='900px'))
gain_output = HTMLMath(layout=Layout(width='900px'))
tau_output = HTMLMath(layout=Layout(width='900px'))
frequency_output = HTMLMath(layout=Layout(width='900px'))
topology_output = HTML(layout=Layout(width='900px'))

parameter_box = VBox([parameter_title, transfer_output, gain_output, tau_output, frequency_output, topology_output], layout=Layout(width='920px'))

# ------------------------------------------------------------
# FIXED FREQUENCY RANGE
# ------------------------------------------------------------

omega_min = 1.0
omega_max = 1.0e6
omega = np.logspace(np.log10(omega_min), np.log10(omega_max), 1600)

# ------------------------------------------------------------
# RESPONSE FUNCTION
# ------------------------------------------------------------

def calculate_response(filter_type, configuration, omega, R1_value, R2_value, R3_value, C1_value):

    jw = 1j * omega

    if configuration == 'Non-inverting':

        K = 1.0 + R2_value / R3_value
        tau = R1_value * C1_value

        if filter_type == 'Low-pass':
            H = K / (1.0 + jw * tau)

        else:
            H = K * jw * tau / (1.0 + jw * tau)

    else:

        K = -R2_value / R1_value

        if filter_type == 'Low-pass':

            tau = R2_value * C1_value
            H = K / (1.0 + jw * tau)

        else:

            tau = R1_value * C1_value
            H = K * jw * tau / (1.0 + jw * tau)

    magnitude = 20.0 * np.log10(np.maximum(np.abs(H), 1e-12))
    phase = np.degrees(np.angle(H))

    return magnitude, phase, K, tau

# ------------------------------------------------------------
# MAIN INTERACTIVE FUNCTION
# ------------------------------------------------------------

def plot_active_filter(filter_type, configuration, R1_kohm, C1_uf, R2_kohm, R3_kohm):

    R1_value = R1_kohm * 1e3
    R2_value = R2_kohm * 1e3
    R3_value = R3_kohm * 1e3
    C1_value = C1_uf * 1e-6

    magnitude, phase, K, tau = calculate_response(filter_type, configuration, omega, R1_value, R2_value, R3_value, C1_value)

    wc = 1.0 / tau
    fc = wc / (2.0 * np.pi)
    gain_db = 20.0 * np.log10(abs(K))

    # --------------------------------------------------------
    # CALCULATED PARAMETERS
    # --------------------------------------------------------

    if configuration == 'Non-inverting':

        gain_output.value = rf"$$K=1+\frac{{R_2}}{{R_3}}={K:.4f}\qquad\qquad20\log_{{10}}|K|={gain_db:.2f}\ \mathrm{{dB}}$$"
        tau_output.value = rf"$$\tau=R_1C_1={tau:.6f}\ \mathrm{{s}}$$"

        if filter_type == 'Low-pass':

            transfer_output.value = r"$$H(s)=\frac{K}{1+sR_1C_1}$$"

            topology_output.value = """
            <div style='font-size:15px;'>
            <b>Non-inverting low-pass:</b>
            R₁ and C₁ determine the cutoff frequency, while R₂ and R₃ determine the gain K.
            </div>
            """

            phase_cutoff = -45.0
            phase_min = -100
            phase_max = 10

        else:

            transfer_output.value = r"$$H(s)=K\frac{sR_1C_1}{1+sR_1C_1}$$"

            topology_output.value = """
            <div style='font-size:15px;'>
            <b>Non-inverting high-pass:</b>
            R₁ and C₁ determine the cutoff frequency, while R₂ and R₃ determine the gain K.
            </div>
            """

            phase_cutoff = 45.0
            phase_min = -10
            phase_max = 100

    else:

        gain_output.value = rf"$$K=-\frac{{R_2}}{{R_1}}={K:.4f}\qquad\qquad20\log_{{10}}|K|={gain_db:.2f}\ \mathrm{{dB}}$$"

        if filter_type == 'Low-pass':

            tau_output.value = rf"$$\tau=R_2C_1={tau:.6f}\ \mathrm{{s}}$$"
            transfer_output.value = r"$$H(s)=-\frac{R_2}{R_1}\frac{1}{1+sR_2C_1}$$"

            topology_output.value = """
            <div style='font-size:15px;'>
            <b>Inverting low-pass:</b>
            R₂ participates in both the gain and the cutoff frequency.
            </div>
            """

            phase_cutoff = 135.0
            phase_min = 80
            phase_max = 190

        else:

            tau_output.value = rf"$$\tau=R_1C_1={tau:.6f}\ \mathrm{{s}}$$"
            transfer_output.value = r"$$H(s)=-\frac{R_2}{R_1}\frac{sR_1C_1}{1+sR_1C_1}$$"

            topology_output.value = """
            <div style='font-size:15px;'>
            <b>Inverting high-pass:</b>
            R₁ participates in both the gain and the cutoff frequency.
            </div>
            """

            phase_cutoff = -135.0
            phase_min = -190
            phase_max = -80

    frequency_output.value = rf"$$\omega_c=\frac{{1}}{{\tau}}={wc:.2f}\ \mathrm{{rad/s}}\qquad\qquad f_c=\frac{{\omega_c}}{{2\pi}}={fc:.2f}\ \mathrm{{Hz}}$$"

    # --------------------------------------------------------
    # NEW FIGURE FOR EACH UPDATE
    # ------------------------------------------------------------

    fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.4))

    ax_mag = axes[0]
    ax_phase = axes[1]

    # --------------------------------------------------------
    # MAGNITUDE RESPONSE
    # ------------------------------------------------------------

    ax_mag.semilogx(omega, magnitude, linewidth=2.0, label='Active filter')
    ax_mag.axvline(wc, linestyle=':', linewidth=1.2)
    ax_mag.axhline(gain_db - 3.0103, linestyle=':', linewidth=1.0)

    ax_mag.set_title('Magnitude Response')
    ax_mag.set_xlabel('Angular Frequency ω (rad/s)')
    ax_mag.set_ylabel('Magnitude (dB)')
    ax_mag.set_xlim(omega_min, omega_max)
    ax_mag.set_ylim(-60, 45)
    ax_mag.grid(True, which='both', linestyle=':', alpha=0.7)
    ax_mag.legend(loc='best')

    # --------------------------------------------------------
    # PHASE RESPONSE
    # --------------------------------------------------------

    ax_phase.semilogx(omega, phase, linewidth=2.0, label='Active filter')
    ax_phase.axvline(wc, linestyle=':', linewidth=1.2)
    ax_phase.axhline(phase_cutoff, linestyle=':', linewidth=1.0)

    ax_phase.set_title('Phase Response')
    ax_phase.set_xlabel('Angular Frequency ω (rad/s)')
    ax_phase.set_ylabel('Phase (deg)')
    ax_phase.set_xlim(omega_min, omega_max)
    ax_phase.set_ylim(phase_min, phase_max)
    ax_phase.grid(True, which='both', linestyle=':', alpha=0.7)
    ax_phase.legend(loc='best')

    # --------------------------------------------------------
    # AXIS FORMAT
    # --------------------------------------------------------

    for ax in axes:

        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.tick_params(axis='x', labelsize=8)
        ax.tick_params(axis='y', labelsize=8)
        ax.set_frame_on(True)

        for spine in ['left', 'right', 'top', 'bottom']:

            ax.spines[spine].set_visible(True)
            ax.spines[spine].set_linewidth(0.8)
            ax.spines[spine].set_clip_on(False)

    fig.subplots_adjust(left=0.08, right=0.97, bottom=0.16, top=0.88, wspace=0.28)

    plt.show()

# ------------------------------------------------------------
# INTERACTIVE WIDGET
# ------------------------------------------------------------

widget_plot = interactive(plot_active_filter, filter_type=filter_radio, configuration=configuration_radio, R1_kohm=R1, C1_uf=C1, R2_kohm=R2, R3_kohm=R3)

# ------------------------------------------------------------
# CONTROL STATE
# ------------------------------------------------------------

def update_control_state(change=None):

    if configuration_radio.value == 'Inverting':
        R3.disabled = True

    else:
        R3.disabled = False

configuration_radio.observe(update_control_state, names='value')

update_control_state()

# ------------------------------------------------------------
# OUTPUT OF INTERACTIVE
# ------------------------------------------------------------

plot_output = widget_plot.children[-1]

# ------------------------------------------------------------
# REMOVE OUTPUT SCROLL BARS
# ------------------------------------------------------------

display(HTML("""
<style>

.jp-OutputArea,
.jp-OutputArea-child,
.jp-OutputArea-output,
.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.output_scroll {
    max-height: none !important;
    height: auto !important;
    overflow: visible !important;
    overflow-y: visible !important;
    overflow-x: visible !important;
}

</style>
"""))

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

display(HTML("<h3>First-Order Active Filter Explorer</h3>"))
display(intro_output)
display(selection_row)
display(control_row)
display(parameter_box)
display(plot_output)